In [1]:
import cortex
import pickle
import numpy as np
import pandas as pd
import nibabel as nib
from collections import Counter

In [2]:
with open("schaefer_parcel_counts.pkl", 'rb') as f:
    parcel_voxel_counts = pickle.load(f)
    
with open('/home/zachkaras/fmri_model/analysis/fir/schaefer_parcel_labels.pkl', 'rb') as f:
    schaefer_parcel_labels = pickle.load(f)

with open("/data/zachkaras/fmri_model_data/intermediate_results/all_results.pkl", 'rb') as f:
    records = pickle.load(f)
   

In [3]:
# low_performing  = [121, 142, 101, 134, 151, 203, 105, 138, 141, 150, 109, 201]
# high_performing = [111, 144, 112, 118, 125, 204, 129, 133, 117, 108, 119, 122]

# poorly_modeled  = [125, 138, 101, 108, 144, 105, 142, 150, 201, 119, 134, 109]
# well_modeled    = [203, 151, 141, 133, 122, 129, 112, 204, 111, 121, 118, 117]

info = {'121' : {'performance': 'low', 'modeled' : 'well'},   '142' : {'performance': 'low', 'modeled': 'poorly'},
        '134' : {'performance': 'low', 'modeled': 'poorly'},  '151' : {'performance': 'low', 'modeled' : 'well'},   '203' : {'performance': 'low', 'modeled' : 'well'},
        '105' : {'performance': 'low', 'modeled': 'poorly'},  '138' : {'performance': 'low', 'modeled': 'poorly'},  '141' : {'performance': 'low', 'modeled' : 'well'},
        '150' : {'performance': 'low', 'modeled': 'poorly'},  '109' : {'performance': 'low', 'modeled': 'poorly'},  '201' : {'performance': 'low', 'modeled': 'poorly'}, 
        '111' : {'performance': 'high', 'modeled' : 'well'},  '144' : {'performance': 'high', 'modeled': 'poorly'}, '112' : {'performance': 'high', 'modeled' : 'well'}, 
        '118' : {'performance': 'high', 'modeled' : 'well'},  '125' : {'performance': 'high', 'modeled': 'poorly'}, '204' : {'performance': 'high', 'modeled' : 'well'}, 
        '129' : {'performance': 'high', 'modeled' : 'well'},  '133' : {'performance': 'high', 'modeled' : 'well'},  '117' : {'performance': 'high', 'modeled' : 'well'}, 
        '108' : {'performance': 'high', 'modeled': 'poorly'}, '119' : {'performance': 'high', 'modeled': 'poorly'}, '122' : {'performance': 'high', 'modeled' : 'well'}}

# no significant correlation between code correct and modeling performance

In [4]:
def translate_to_region_names(parcel_num, schaefer_labels):
        parcel_num = int(parcel_num)
        if parcel_num <= 200:
            region = f"Left {schaefer_labels['left'][parcel_num]}"
        else:
            region = f"Right {schaefer_labels['right'][parcel_num]}"
        return region

def get_top_regions(parcel_dict, n_regions=5):
    parcel_dict = dict(sorted(parcel_dict.items(), key= lambda x: x[1], reverse=True))
    return {region : val for i,(region,val) in enumerate(parcel_dict.items()) if i < n_regions}


def compute_jaccard(df):
    list_of_sets = [set(row['top_five_regions'].keys()) for i,row in df.iterrows()]
    intersection = set.intersection(*list_of_sets)
    union = set.union(*list_of_sets)
    return len(intersection) / len(union)

In [5]:
# for vip in important_participants:

mask = ((records['look_ahead'] == 'look_ahead_by_0') &
        (records['ndelays'] == 'ndelays_10') &
        (records['model'] == 'deepseek_6b'))
for (model, task, participant), df in records[mask].groupby(['model', 'task', 'participant']):
    
    if participant not in info.keys():
        print(f'Skipping {participant}')
        continue

    temp = df['top_parcels'].apply(lambda row: [translate_to_region_names(el, schaefer_parcel_labels) for el in row])
    
    df['top_HarvOx_regions'] = temp.apply(lambda row: dict(Counter(row)))
    df['top_five_regions'] = df['top_HarvOx_regions'].apply(lambda row: get_top_regions(row, n_regions=5))
    
    filtered = df[['layer', 'top_five_regions']].copy()
    filtered = filtered.sort_values('layer', key=lambda col: col.map(lambda x: int(x.split('_')[1])))
    
    jaccard_index = compute_jaccard(filtered)
#     print(model, task, participant, type(participant), jaccard_index)
    info[participant][f'jaccard-{task}'] = jaccard_index

df = pd.DataFrame.from_dict(info, orient='index')

Skipping 101
Skipping 130


In [6]:
from scipy import stats

for metric in ['jaccard-code', 'jaccard-prose']:
    low  = df[df['performance'] == 'low'][metric].dropna()
    high = df[df['performance'] == 'high'][metric].dropna()
    t, p = stats.ttest_ind(low, high)
    print(f"{metric} | low vs high performance: t={t:.3f}, p={p:.3f}")

for metric in ['jaccard-code', 'jaccard-prose']:
    well   = df[df['modeled'] == 'well'][metric].dropna()
    poorly = df[df['modeled'] == 'poorly'][metric].dropna()
    t, p = stats.ttest_ind(well, poorly)
    print(f"{metric} | well vs poorly modeled:  t={t:.3f}, p={p:.3f}")
    
paired = df[['jaccard-code', 'jaccard-prose']].dropna()
t, p = stats.ttest_rel(paired['jaccard-code'], paired['jaccard-prose'])
print(f"Code vs. Prose Jaccard Scores:  t={t:.3f}, p={p:.3f}")

jaccard-code | low vs high performance: t=-0.190, p=0.851
jaccard-prose | low vs high performance: t=2.071, p=0.051
jaccard-code | well vs poorly modeled:  t=1.548, p=0.137
jaccard-prose | well vs poorly modeled:  t=-0.008, p=0.993
Code vs. Prose Jaccard Scores:  t=0.869, p=0.394


# Brain plots

In [7]:
atlas_base_path = "/home/zachkaras/fmri_model/analysis/pipeline/atlases"

# read in 2d mni mask
mask = nib.load(f"{atlas_base_path}/MNI152_T1_2mm_brain_mask.nii.gz")
og_shape = mask.shape
mask = mask.get_fdata().flatten()
brain_idx = np.where(mask>0)[0]

atlas = nib.load(f"{atlas_base_path}/Schaefer2018_400Parcels_7Networks_order_FSLMNI152_2mm.nii.gz")
atlas_vec = atlas.get_fdata().flatten()
atlas_only_brain = atlas_vec[brain_idx] # contains the schaefer parcel numbers
cortex_vx = np.where(atlas_only_brain != 0)[0]
schaefer_voxels = atlas_only_brain[cortex_vx]

# Making empty templates to save output
empty_schaefer = np.zeros(atlas_only_brain.shape)
empty_mni = np.zeros(atlas_vec.shape)


def convert_to_nifti(values):
    # working backwards to save correlation values as voxels in MNI space
    new_schaefer = empty_schaefer.copy()
    new_mni = empty_mni.copy()
    new_schaefer[cortex_vx] = values
    new_mni[brain_idx] = empty_schaefer
    result_brain = np.reshape(empty_mni, og_shape)

    # Saving results
    nifti_result = nib.Nifti1Image(result_brain, affine=atlas.affine, header=atlas.header)
    nib.save(nifti_result, "test_plotting.nii.gz")
    return result_brain, nifti_result

In [8]:
# I'd like to compare high vs. low performance and well vs. poorly modeled
# I need to create four stat maps

def increment_parcels(top_parcels, schaefers, participant, info):
    performance = info[participant]['performance']
    modeled = info[participant]['modeled']
    
    for p in top_parcels:
        schaefers[performance][p] += 1
        schaefers[modeled][p] += 1
    return schaefers
    

def make_schafer_data(curr_task):
    mask = ((records['look_ahead'] == 'look_ahead_by_0') &
            (records['ndelays'] == 'ndelays_10') &
            (records['model'] == 'deepseek_6b'))
    
    schaefers = {'well'  : np.zeros(401),
                'poorly' : np.zeros(401),
                'low'    : np.zeros(401),
                'high'   : np.zeros(401)}

    for (model, task, participant), df in records[mask].groupby(['model', 'task', 'participant']):
        if task != curr_task:
            continue
        
        if participant not in info.keys():
            continue
        
        df['parcel_counts'] = df['top_parcels'].apply(lambda row: dict(Counter(row)))
        df['parcel_counts'] = df['parcel_counts'].apply(lambda row: get_top_regions(row, n_regions=10))

        top_parcels = set(df['parcel_counts'].explode()) # set of all the regions that are among the top 10 schaefer parcels across the layers for a participant
        top_parcels = {int(i) for i in top_parcels}
        schaefers = increment_parcels(top_parcels, schaefers, participant, info)
    return schaefers

def save_schaefer_maps(schaefers, task):
    # find indices of voxels corresponding to each schaefer parcel
    # is there a way to parallelize?
    for group, lookup in schaefers.items():        
        result = lookup[schaefer_voxels.astype(int)]
        
        npy_brain, nifti_brain = convert_to_nifti(result)
        npy_brain = npy_brain.transpose(2,1,0)
    
        vol = cortex.Volume(
        npy_brain,
        subject='fsaverage',
        xfmname='mni2py2',
        )
        cortex.webshow(vol)
        break
    
    
    # set value of those voxels equal to number in dictionaries


In [9]:
code_schaefers = make_schafer_data('code')
prose_schaefers = make_schafer_data('prose')
# save_schaefer_maps(code_schaefers, 'code')
    # break

In [10]:
with open("/tank/home/zachkaras/code_schaefers.pkl", 'wb') as f:
    pickle.dump(code_schaefers, f)

with open("/tank/home/zachkaras/prose_schaefers.pkl", 'wb') as f:
    pickle.dump(prose_schaefers, f)

In [ ]:
with open("/tank/home/zachkaras/code_schaefers.pkl", 'wb') as f:
    pickle.dump(code_schaefers, f)

with open("/tank/home/zachkaras/prose_schaefers.pkl", 'wb') as f:
    pickle.dump(prose_schaefers, f)